# CATS — Contextual Ambiguity & Trust Scoring: 5-minute demo

**Trust intelligence for OSINT sources — not fact-checking, but source reliability over time.**

CATS analyses the *behavioural patterns* of a source (narrative coherence, sentiment
volatility, temporal silence, gaming signs) and returns a transparent, explainable
ordinal trust score. This notebook uses `cats.lite`: the full signal pipeline as a
plain library call — no database, no Redis, no API keys.

Repo: https://github.com/Leapfrog-LSA/CATS-Contextual-Ambiguity-Trust-Scoring


In [ ]:
# Install CATS (lite surface) straight from GitHub
%pip install -q git+https://github.com/Leapfrog-LSA/CATS-Contextual-Ambiguity-Trust-Scoring.git
# Optional but recommended: the Italian NER model for full-fidelity coherence
# (without it, the coherence signal degrades to a neutral value).
!python -m spacy download it_core_news_lg -q


## Score a well-behaved newsroom

A steady publishing rhythm, consistent narrative, no manipulation patterns.


In [ ]:
from cats.lite import score

newsroom = [
    {"timestamp": "2026-06-01T08:00:00Z", "text": "Il governo presenta il piano economico alla Camera."},
    {"timestamp": "2026-06-01T13:00:00Z", "text": "I sindacati chiedono modifiche al piano del governo."},
    {"timestamp": "2026-06-02T09:00:00Z", "text": "Il parlamento avvia la discussione sulla legge di bilancio."},
    {"timestamp": "2026-06-02T18:00:00Z", "text": "La commissione bilancio approva i primi emendamenti."},
    {"timestamp": "2026-06-03T08:30:00Z", "text": "Il ministro dell'economia illustra la manovra in conferenza stampa."},
    {"timestamp": "2026-06-03T15:00:00Z", "text": "Le opposizioni presentano una mozione sulla manovra economica."},
]

result = score(newsroom, source_type="news")
print(f"trust_score = {result['trust_score']}  band = {result['band']}")
result["signals"]


## Score a suspicious source

Sporadic bursts after long gaps, repetitive sensationalist text — the behavioural
fingerprint the July 2026 calibration found on documented disinformation feeds
(see `docs/calibration_findings_2026-07.md`).


In [ ]:
suspicious = [
    {"timestamp": "2026-01-05T02:00:00Z", "text": "CLAMOROSO: la verità che nessuno vi dice sui vaccini!!!"},
    {"timestamp": "2026-01-05T02:05:00Z", "text": "CLAMOROSO: la verità che nessuno vi dice sull'economia!!!"},
    {"timestamp": "2026-01-05T02:10:00Z", "text": "CLAMOROSO: la verità che nessuno vi dice sul clima!!!"},
    {"timestamp": "2026-03-20T23:00:00Z", "text": "SVEGLIA!!! Vi nascondono tutto, condividi prima che lo cancellino!"},
    {"timestamp": "2026-06-28T04:00:00Z", "text": "SVEGLIA!!! Vi nascondono tutto sul nuovo scandalo, condividi!"},
]

bad = score(suspicious, source_type="news")
print(f"trust_score = {bad['trust_score']}  band = {bad['band']}  review = {bad['requires_human_review']}")
bad["signals"]


## Why did it get that score?

Every score decomposes into per-signal contributions on a common reliability axis
(negative-polarity signals — volatility, silence, gaming — are inverted as `100 − value`).
This is the same payload the production `/explain` GDPR endpoint returns.


In [ ]:
import json
print(json.dumps(bad["explanation"], indent=2, ensure_ascii=False))


## Caveats (read before citing numbers)

- Scores are **ordinal rankings**, not probabilities — never a sole basis for automated decisions (WP 4.3).
- The default NLP is Italian-optimised; enable the multilingual Sentence-BERT coherence
  backend with `COHERENCE_BACKEND=sbert` (`pip install \"cats-scoring[sbert]\"`).
- Current empirical validation: 50 RSS-labelled sources, full-dataset concordance 0.78 —
  honest numbers and their limits in
  [`docs/calibration_findings_2026-07.md`](https://github.com/Leapfrog-LSA/CATS-Contextual-Ambiguity-Trust-Scoring/blob/main/docs/calibration_findings_2026-07.md).
